# 15. Real Paper Reproduction Case Study — CARE / CSBDeep

**Paper:** Weigert et al., *Content-aware image restoration: pushing the limits of fluorescence microscopy*, Nature Methods (2018)  
**Authors' code:** `https://github.com/CSBDeep/CSBDeep`  
**Goal:** reproduce one CARE experiment correctly, understand where the paper is implemented in code, then decide **REUSE / ADAPT / REIMPLEMENT / REJECT**.

> Important: CARE contains several restoration demonstrations. Do not say "I reproduced CARE" until you name the exact experiment, dataset, preprocessing, model configuration, and metric you reproduced.

## Mind map

```mermaid
mindmap
  root((CARE paper to code))
    Paper
      Input and target
      Optical acquisition
      Registration
      Loss and metric
    Repository
      License
      Release or commit
      Examples
      Config
      Model code
    Reproduction
      Freeze environment
      Run authors demo
      Tiny training
      Full selected run
      Compare figure or metric
    Adaptation
      New microscope
      New sampling
      New noise
      New target
    Scientific QC
      Weak structures
      Resolution
      Intensity fidelity
      Hallucination
```

## Why choose CARE?

It is an ideal optical-imaging case because the scientific question is tied to acquisition: can lower-photon or lower-quality fluorescence data be restored toward a higher-quality reference without increasing phototoxicity/exposure?

The key supervision structure is:

```text
low-quality acquisition  ---> network ---> higher-quality paired reference
```

This is only scientifically meaningful when the pair is trustworthy.


## Step 1 — Build the paper specification before touching GitHub

Fill this table from the paper + supplement:

| Item | What you must know |
|---|---|
| Input | modality, axes, bit depth, pixel size, exposure/SNR |
| Target | how the cleaner reference was acquired |
| Pairing | same FOV? same time? registration method? rejected pairs? |
| Split | specimen/field/stack level, not merely random patches |
| Preprocessing | clipping, normalization, resizing, background correction |
| Patches | patch size, sampling rule, augmentation |
| Model | CARE config / U-Net-like structure |
| Loss | exact selected experiment loss |
| Optimizer | LR, schedule, batch size, epochs |
| Checkpoint | best validation? last? specific weights? |
| Metric | exact definition and aggregation |
| Baseline | raw input + classical comparison |
| Scientific QC | dim structures, morphology, resolution, intensity |

**Stop condition:** if the target pairing or split is unclear, do not start a long training run.

## Step 2 — Audit the authors' repository

The official CSBDeep repository is a Keras/TensorFlow toolbox for CARE and is BSD-3-Clause licensed. It has continued to evolve since the 2018 paper. Therefore:

```text
current repository != automatically the historical paper environment
```

Record:

```text
paper DOI
repository URL
git commit/tag
CSBDeep version
Python
TensorFlow/Keras
CUDA/cuDNN
GPU
dataset version/checksum
example notebook used
```

Use two tracks:

**Track A — reproduction:** preserve the historical/reference stack; do not modernize while debugging.  
**Track B — research implementation:** only after Track A works, port/adapt to PyTorch or a maintained stack.


## Step 3 — Reproduction ladder

```mermaid
flowchart TD
  A[Clone exact code/release] --> B[Import smoke test]
  B --> C[Load authors example]
  C --> D[Run inference]
  D --> E[Verify axes + normalization]
  E --> F[Tiny training run]
  F --> G[Full selected example]
  G --> H[Compare metric/figure]
  H --> I{Match?}
  I -->|No| J[Discrepancy log]
  J --> B
  I -->|Yes| K[Adapt to own data]
```

Suggested shell strategy:

```bash
git clone https://github.com/CSBDeep/CSBDeep.git
cd CSBDeep
git rev-parse HEAD
git status

conda create -n care-repro python=<version_required_by_selected_release>
conda activate care-repro

# Install dependencies matching the selected release.
# Do not blindly upgrade TensorFlow/Keras during reproduction.
```

### Source-map strategy

When reading the code, map every scientific operation:

| Paper concept | Code to locate |
|---|---|
| model/config | CARE / Config |
| patch generation | data patch utilities |
| normalization | training/prediction normalizer |
| training | CARE train / Keras fit |
| checkpoint | callback + weight loading |
| tiled inference | prediction utilities |
| metric | paper/example evaluation code |

Your question should always be: **where is this paper claim implemented?**


## Step 4 — A self-contained teaching reproduction of the supervised idea

The next code is **not the original CARE architecture** and does **not claim a paper metric**. It is a controlled PyTorch example showing how to conceptualize a paired optical-restoration problem before using the authors' full code.

We simulate:
1. latent fluorescence structure;
2. low-photon input;
3. high-photon reference;
4. a small residual CNN;
5. training against the paired target;
6. comparison with the raw-input baseline.


In [ ]:
import math, numpy as np, torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(7)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

def scene(size=48, seed=0):
    g = torch.Generator().manual_seed(seed)
    y,x = torch.meshgrid(torch.linspace(-1,1,size), torch.linspace(-1,1,size), indexing="ij")
    img = torch.zeros(size,size)
    for _ in range(10):
        cx = torch.rand(1,generator=g).item()*1.6-0.8
        cy = torch.rand(1,generator=g).item()*1.6-0.8
        s  = 0.03 + 0.08*torch.rand(1,generator=g).item()
        a  = 0.3  + 0.7*torch.rand(1,generator=g).item()
        img += a*torch.exp(-((x-cx)**2+(y-cy)**2)/(2*s*s))
    return img/img.max().clamp_min(1e-8)

def acquire(clean, photons):
    return torch.poisson(clean*photons)/photons

def dataset(n=16):
    X,Y=[],[]
    for i in range(n):
        clean=scene(seed=100+i)
        X.append(acquire(clean,12)[None])
        Y.append(acquire(clean,250)[None])
    return torch.stack(X), torch.stack(Y)

X,Y=dataset()
Xtr,Ytr,Xv,Yv = X[:12],Y[:12],X[12:],Y[12:]
print(Xtr.shape, Xv.shape)


In [ ]:
class TinyRestorer(nn.Module):
    def __init__(self,w=20):
        super().__init__()
        self.f = nn.Sequential(
            nn.Conv2d(1,w,3,padding=1), nn.ReLU(),
            nn.Conv2d(w,w,3,padding=1), nn.ReLU(),
            nn.Conv2d(w,1,3,padding=1)
        )
    def forward(self,x):
        return x + self.f(x)

model=TinyRestorer().to(device)
opt=torch.optim.Adam(model.parameters(),lr=2e-3)
xt,yt=Xtr.to(device),Ytr.to(device)

for step in range(50):
    pred=model(xt)
    loss=F.l1_loss(pred,yt)
    opt.zero_grad(); loss.backward(); opt.step()

def psnr(a,b):
    m=torch.mean((a-b)**2).item()
    return 10*math.log10(1/max(m,1e-12))

with torch.no_grad():
    pv=model(Xv.to(device)).cpu().clamp(0,1)

raw=np.mean([psnr(Xv[i],Yv[i]) for i in range(len(Xv))])
net=np.mean([psnr(pv[i],Yv[i]) for i in range(len(Xv))])
print("raw baseline PSNR:",round(raw,2))
print("network PSNR:",round(net,2))


## Step 5 — What this example proves / does not prove

**It proves:** paired tensor construction, supervised restoration logic, baseline comparison, and a debugging path.

**It does not prove:** the CARE architecture, the original CARE loss/config, the paper's numerical result, real detector noise, or biological validity.

To move from teaching example to faithful reproduction, replace **one component at a time**:

```text
synthetic paired data -> authors example data
toy network           -> CARE model/config
our normalization     -> authors normalization
our loss              -> exact experiment loss
our metric            -> exact authors metric
```

Run a smoke test after every replacement.

## Step 6 — Optical-imaging adaptation decisions

**Fluorescence low/high exposure:** strong conceptual match if pairs are aligned.  
**LSFM:** validate z anisotropy, PSF, bleaching, and volume registration.  
**MUSE / virtual histology:** restoration and virtual staining are not the same task; target definition/registration dominate.  
**OCT:** do not assume fluorescence noise logic applies to coherent speckle. The target and physics require separate justification.

## Final decision table

| Reproduction result | Action |
|---|---|
| authors demo matches + your task closely matches | REUSE as reference, then validate |
| authors demo matches but acquisition differs | ADAPT with controlled ablations |
| concept fits but framework is unsuitable | REIMPLEMENT after reproduction |
| paired target is unreliable | REJECT / redesign supervision |
| prettier output removes weak structures | REJECT that model/setting |

### Reproduction record

```text
selected CARE experiment:
repo commit/tag:
environment:
data/checksum:
input/target axes:
registration:
normalization:
patch extraction:
model config:
loss:
optimizer/LR:
checkpoint:
authors metric:
my metric:
difference:
failure cases:
final decision:
```
